<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Demo: Role-Based Agent Design </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

**Purpose:**  
Demonstrate how to design **role-based agents** with specialized responsibilities, custom prompts, and basic inter-agent communication.

This notebook shows how to move from generic agents to clearly defined roles (Planner, Researcher, Executor) — a foundational pattern in multi-agent systems.

**Key Topics Covered:**
- Defining clear agent roles
- Role-specific system prompts
- Basic message passing between agents
- Logging and tracing inter-agent communication
- Benefits of role specialization

---

## 1. Setup

In [1]:
!pip install -q --force-reinstall "langchain>=0.1.0,<1.0" "langchain-core==0.3.*" "langchain-openai==0.2.*" "langchain-community==0.3.*" langsmith duckduckgo-search ddgs faiss-cpu beautifulsoup4 requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 744.6/744.6 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18

In [2]:
import os
from getpass import getpass
from typing import Dict, List

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("Setup complete!")

Enter your OpenAI API key: ··········
Setup complete!


## 2. Defining Agent Roles

We will create a simple **Travel Planning Team** with three specialized roles:

- **Planner Agent**: Breaks down the task and creates a plan
- **Researcher Agent**: Gathers information using search
- **Executor Agent**: Performs calculations and produces the final output

In [3]:
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_openai import ChatOpenAI
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub
from langchain_core.prompts import ChatPromptTemplate

search_tool = DuckDuckGoSearchRun(name="Web_Search")

@tool
def calculate_math(expression: str) -> str:
    """Perform mathematical calculations."""
    try:
        allowed = {"__builtins__": {}}
        return str(eval(expression, allowed, {}))
    except Exception as e:
        return f"Error: {str(e)}"

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("Tools and LLM ready")

Tools and LLM ready


## 3. Role-Specific Prompts

In [4]:
from langsmith import Client

client = Client()

# Load the standard ReAct prompt
react_prompt = client.pull_prompt("hwchase17/react",
    dangerously_pull_public_prompt=True)

# Role-specific instructions (we will inject these into the prompt)
planner_instruction = """You are a Planner Agent. Your only job is to break down the user's task into clear, logical steps.
You do NOT search or calculate. You only create a plan and specify which agent should do what."""

researcher_instruction = """You are a Researcher Agent. Your only job is to gather information using the Web_Search tool.
Provide clear, concise summaries of what you find. Do not perform calculations."""

executor_instruction = """You are an Executor Agent. Your job is to use the calculate_math tool to perform calculations
and produce a clean, professional final summary."""

print("Base ReAct prompt loaded")

Base ReAct prompt loaded


## 4. Creating Role-Based Agents

In [5]:
# Researcher Agent
researcher_prompt = react_prompt.partial(
    system_message=researcher_instruction
)

researcher_agent = create_react_agent(
    llm=llm,
    tools=[search_tool],
    prompt=researcher_prompt
)

researcher_executor = AgentExecutor(
    agent=researcher_agent,
    tools=[search_tool],
    verbose=False,
    max_iterations=6,
    handle_parsing_errors=True
)

# Executor Agent - Stronger prompt to force tool usage
executor_instruction = """You are an Executor Agent. Your ONLY job is to use the calculate_math tool to perform calculations.

You will be given data from the Researcher. Extract the numbers carefully and use the calculate_math tool for ALL calculations.
Do not assume values. Do not make up numbers. Always use the tool for math.

Final output must be clean and professional."""

executor_prompt = react_prompt.partial(
    system_message=executor_instruction
)

executor_agent = create_react_agent(
    llm=llm,
    tools=[calculate_math],
    prompt=executor_prompt
)

executor_executor = AgentExecutor(
    agent=executor_agent,
    tools=[calculate_math],
    verbose=False,
    max_iterations=5,
    handle_parsing_errors=True
)
# -----------------------------------------------------------------------------
# STUDENT TASK 2: Create the Planner Agent and its Executor
# -----------------------------------------------------------------------------
# Create the `planner_agent` and `planner_executor` similar to the Researcher and Executor.
# Remember:
# - The Planner Agent uses the `planner_instruction` you defined in the previous cell.
# - The Planner Agent does NOT use any tools directly for its planning task.
# - Use `create_react_agent` and `AgentExecutor`.
# - Set `verbose=False`, `max_iterations` to a reasonable number (e.g., 3), and `handle_parsing_errors=True`.

# Your code for Planner Agent creation goes below this comment:

planner_prompt = react_prompt.partial(
    system_message=planner_instruction
)
planner_agent = create_react_agent(
    llm=llm,
    tools=[], # Planner does not use tools
    prompt=planner_prompt
)
planner_executor = AgentExecutor(
    agent=planner_agent,
    tools=[], # Planner does not use tools
    verbose=False,
    max_iterations=3, # A planner usually doesn't need many iterations
    handle_parsing_errors=True
)
print("Researcher and Executor Agents created!")

Researcher and Executor Agents created!


## 5. Basic Inter-Agent Communication + Logging

In [6]:
def run_role_based_travel_planner(user_query: str):
    print("Starting Multi-Agent Travel Planning System\n")
    interactions = []

    # Step 1: Planner (simple hardcoded plan for Day 1)
    print("Planner Agent: Creating plan...")
    plan = """1. Researcher: Gather flight price (Riyadh → Istanbul round-trip in USD),
             average hotel price per night in Istanbul (USD), and current USD to SAR exchange rate.
2. Executor: Calculate total budget in SAR (flight + 7 nights hotel) and flight percentage of total."""

    interactions.append({"from": "Planner", "to": "System", "message": plan})
    print(plan + "\n")

    # Step 2: Researcher Agent - Make input very clear
    print("Researcher Agent: Gathering information...")
    research_input = """Extract these exact values:
- Cheapest round-trip flight price from Riyadh to Istanbul in USD
- Average hotel price per night in Istanbul in USD
- Current USD to SAR exchange rate

Return only the key numbers and short summary."""

    research_result = researcher_executor.invoke({"input": research_input})
    research_data = research_result["output"]

    interactions.append({"from": "Researcher", "to": "Executor", "message": research_data[:300] + "..."})
    print("Research completed.\n")

    # Step 3: Executor Agent - Give very structured input
    print("Executor Agent: Performing calculations...")
    executor_input = f"""Use the following research data:

{research_data}

Tasks:
1. Calculate total cost in SAR: (round-trip flight USD × exchange rate) + (hotel price per night USD × 7 × exchange rate)
2. Calculate what percentage the flight cost is of the total budget.

Use the calculate_math tool for every calculation. Give a clean final summary with all numbers in SAR."""

    executor_result = executor_executor.invoke({"input": executor_input})

    interactions.append({"from": "Executor", "to": "User", "message": "Final calculation complete"})

    # Final Output
    print("\n" + "="*70)
    print("FINAL OUTPUT FROM EXECUTOR AGENT:")
    print(executor_result["output"])

    print("\n" + "="*70)
    print("INTER-AGENT COMMUNICATION LOG:")
    for idx, msg in enumerate(interactions, 1):
        print(f"{idx}. {msg['from']} → {msg['to']}")

    return executor_result["output"]

# Run the demo
result = run_role_based_travel_planner("Plan a budget trip from Riyadh to Istanbul")

Starting Multi-Agent Travel Planning System

Planner Agent: Creating plan...
1. Researcher: Gather flight price (Riyadh → Istanbul round-trip in USD),
             average hotel price per night in Istanbul (USD), and current USD to SAR exchange rate.
2. Executor: Calculate total budget in SAR (flight + 7 nights hotel) and flight percentage of total.

Researcher Agent: Gathering information...
Research completed.

Executor Agent: Performing calculations...

FINAL OUTPUT FROM EXECUTOR AGENT:
Agent stopped due to iteration limit or time limit.

INTER-AGENT COMMUNICATION LOG:
1. Planner → System
2. Researcher → Executor
3. Executor → User


## 6. Reflection Questions

1. How does giving each agent a clear role improve the system?
2. What are the advantages of role-specific prompts?
3. How did we simulate communication between agents?
4. In what situations would you need more than 3 roles?
5. What improvements would you make to the communication logging?
